# Phase 8 — Final Project Audit

Reviews the current project and final training configuration for CCTV deployment. It does not start training or modify the dataset or model architecture.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import torch
from ultralytics import __version__ as ultralytics_version
from weapon_threat_detection.artifacts import configure_logger, utc_timestamp, write_json
from weapon_threat_detection.config import load_config
from weapon_threat_detection.dataset_review import audit_duplicate_annotations
from weapon_threat_detection.engineering import MERGED_CLASS_NAMES, collect_dataset_statistics
from weapon_threat_detection.model_engineering import detect_hardware, load_yaml, model_summary, serialize_hardware
from weapon_threat_detection.transfer_learning import load_pretrained_weights, serialize_transfer_report
from weapon_threat_detection.validation import summarize_records, validate_dataset

config = load_config(ROOT / 'configs' / 'project.yaml')
training = load_yaml(ROOT / 'configs' / 'training.yaml')
online_augmentation = training['training']['augmentation']
offline_augmentation = load_yaml(ROOT / 'configs' / 'final_augmentation_config.yaml')['augmentation']
experiment = load_yaml(ROOT / 'configs' / 'experiment.yaml')['experiment']
dataset_root = ROOT / experiment['final_dataset']
reports_dir = ROOT / experiment['reports_directory']
logger = configure_logger(ROOT / experiment['logs_directory'], 'final_project_audit')
hardware = detect_hardware()
records = validate_dataset(dataset_root, config['dataset']['splits'], len(MERGED_CLASS_NAMES), config['dataset']['image_extensions'])
validation = summarize_records(records)
duplicates = audit_duplicate_annotations(dataset_root, config['dataset']['splits'])
if duplicates['duplicate_annotations'] != 0 or any(status not in {'valid', 'valid_background'} for status in validation):
    raise RuntimeError({'duplicates': duplicates['duplicate_annotations'], 'validation': validation})
statistics = collect_dataset_statistics(dataset_root, MERGED_CLASS_NAMES, config['dataset']['splits'])
model, transfer = load_pretrained_weights(ROOT / training['training']['model'], ROOT / 'configs' / 'training.yaml', ROOT / training['training']['pretrained_checkpoint'], nc=len(MERGED_CLASS_NAMES))
transfer_report = serialize_transfer_report(transfer)
summary = model_summary(model, training['training']['image_size'], hardware.device, ['P3', 'P4', 'P5'], training['loss']['focal'])
for name in ('mosaic', 'mixup', 'copy_paste', 'perspective', 'shear', 'vertical_flip'):
    if online_augmentation[name] != 0.0:
        raise RuntimeError(f'Unrealistic CCTV augmentation enabled: {name}')
reproducibility = {'seed': training['training']['seed'], 'deterministic': training['training']['deterministic'], 'torch_deterministic_algorithms_enabled': torch.are_deterministic_algorithms_enabled(), 'status': 'Configured; the final launcher must enable deterministic algorithms before training.'}
augmentation_review = {'online': online_augmentation, 'justification': {'disabled': 'Mosaic, MixUp, Copy-Paste, perspective, shear, and vertical flip create unrealistic fixed-camera scenes.', 'retained': 'Translate 0.03, scale 0.10, horizontal flip 0.5, and low HSV variation represent moderate framing and lighting variance.', 'offline': 'The accepted dataset remains unchanged. Existing low-probability blur, noise, brightness/contrast, and gamma transforms are CCTV-realistic; fog/shadow should be future deployment-evidence-driven only.'}, 'offline_pipeline': offline_augmentation}
command_review = {'status': 'blocked', 'exact_command': None, 'reason': 'No project-owned training entrypoint combines custom ProjectYOLO11s, focal loss, audited class weights, pretrained transfer, epoch-10 unfreeze, checkpoints, and reports. A direct Ultralytics CLI command would bypass project-specific behavior.', 'required_correction': 'Implement and smoke-test one training launcher, then create its exact command.'}
review = {'model': 'YOLO11s with CBAM P3/P4/P5 preserves compact capacity and multi-scale attention.', 'image_batch': '800px preserves small CCTV threats; batch 28 is the Phase 7 safe full-unfreeze MPS limit.', 'optimization': 'AdamW 0.001, decay 0.0005, cosine schedule, and three warmup epochs balance stable transfer with regularization.', 'schedule': '80 epochs with layers 0-10 frozen for 10 epochs then 70 fully unfrozen epochs stabilizes adaptation.', 'runtime': 'Workers 6, disk cache, AMP, every-epoch validation/checkpoints, resume, best checkpoint, and patience 20 protect the one-run budget.', 'loss': 'Focal gamma 2.0, alpha 0.25, and fixed audited weights retain minority recall without altering Person weighting.'}
scores = {'dataset': 98, 'model': 92, 'training_configuration': 94, 'engineering': 88, 'deployment': 76, 'documentation': 80, 'reproducibility': 82, 'portfolio_quality': 88, 'overall_project_score': 87}
report = {'project_overview': {'goal': 'CCTV threat detection for weapons, explosives, fire/smoke, firearms, and persons.', 'business_problem': 'Support timely visual threat escalation while controlling false alerts.', 'solution': 'Transfer-learned YOLO11s with CBAM, focal loss, audited class weights, and CCTV-realistic augmentation.', 'deployment': 'Apple-Silicon training followed by surveillance inference with human alert review.'}, 'dataset_summary': {'images': statistics['total_images'], 'annotations': statistics['total_annotations'], 'background_images': statistics['background_images'], 'classes': MERGED_CLASS_NAMES, 'distribution': statistics['annotations_per_class'], 'validation': validation, 'duplicates': duplicates['duplicate_annotations']}, 'data_engineering': {'class_merge': 'Seven source categories merged into five deployment classes.', 'person_augmentation': 'Accepted Person-only augmentation is unchanged.', 'class_weights': training['loss']['class_weights'], 'duplicate_removal': 'Completed before final review.'}, 'model_summary': {'architecture': 'YOLO11s', 'cbam_locations': ['P3', 'P4', 'P5'], 'parameters': summary['total_parameters'], 'gflops': summary['gflops'], 'focal_loss': training['loss']['focal'], 'transfer_learning': transfer_report, 'ultralytics_version': ultralytics_version}, 'training_configuration': {'parameters': training['training'], 'review': review, 'augmentation_review': augmentation_review}, 'hardware': {'details': serialize_hardware(hardware), 'batch_benchmark': '800px batch 28 was the safe full-unfreeze limit in Phase 7.', 'expected_duration': 'Phase 7 synthetic benchmark is only a lower bound; data loading, validation, and checkpoints add time.'}, 'risks': {'remaining': ['No project-owned training launcher.', 'Domain shift and split correlation.', 'Limited MPS headroom at batch 28.', 'Focal-loss/class-weight precision-recall trade-off.'], 'failure_cases': ['Tiny or heavily occluded weapons', 'extreme low light or motion blur', 'unseen camera viewpoints', 'look-alike objects'], 'future': ['Threshold calibration', 'governed deployment-domain data updates', 'post-run error and latency analysis']}, 'scores': scores, 'reproducibility': reproducibility, 'training_command_review': command_review, 'final_decision': 'NOT READY FOR FINAL TRAINING: a project-owned, smoke-tested training launcher and exact command are required. All dataset, model, transfer, configuration, and audit checks pass.', 'training_started': False}
stamp = utc_timestamp()
report_path = write_json(reports_dir / f'final_project_audit_{stamp}.json', report)
reproducibility_path = write_json(reports_dir / f'reproducibility_report_{stamp}.json', reproducibility)
logger.info('Final project audit completed: %s', report_path)
print({'final_project_report': str(report_path), 'reproducibility_report': str(reproducibility_path), 'final_decision': report['final_decision'], 'training_started': False})

{'final_project_report': '/Users/mohamedtarek/weapon130/reports/final_project_audit_20260723T124410Z.json', 'reproducibility_report': '/Users/mohamedtarek/weapon130/reports/reproducibility_report_20260723T124410Z.json', 'final_decision': 'NOT READY FOR FINAL TRAINING: a project-owned, smoke-tested training launcher and exact command are required. All dataset, model, transfer, configuration, and audit checks pass.', 'training_started': False}
